# Dataset and DataLoader

In the training loop, you don't pass all data at once — you pass it in **batches**.
`Dataset` defines how to get one sample. `DataLoader` batches them, shuffles, and loads in parallel.

**Java analogy:** Dataset = `Iterator<Sample>`, DataLoader = a thread-pooled batch producer.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# Custom Dataset — always implement:
#   __len__  → how many samples?
#   __getitem__ → return one sample by index

class RegressionDataset(Dataset):
    def __init__(self, n_samples=1000):
        torch.manual_seed(42)
        self.X = torch.randn(n_samples, 4)
        self.y = self.X[:, 0] * 2 + self.X[:, 1] * (-1) + 0.5   # known formula

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]   # return one (features, label) pair


dataset = RegressionDataset()
print('Dataset size:', len(dataset))
x_sample, y_sample = dataset[0]
print('Sample X:', x_sample)
print('Sample y:', y_sample)

In [ ]:
# DataLoader — wraps Dataset, handles batching, shuffling, parallel loading
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,       # shuffle each epoch (important for training)
    num_workers=0       # parallel loading workers (0 = main thread, safe on Windows)
)

print('Number of batches:', len(loader))   # 1000 / 32 = ~32 batches

# Peek at one batch
X_batch, y_batch = next(iter(loader))
print('Batch X shape:', X_batch.shape)   # [32, 4]
print('Batch y shape:', y_batch.shape)   # [32]

In [ ]:
import torch.nn as nn

# Full training loop with DataLoader
torch.manual_seed(42)

model     = nn.Linear(4, 1)
loss_fn   = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(5):
    total_loss = 0
    for X_batch, y_batch in loader:   # iterate over all batches
        y_pred = model(X_batch).squeeze()
        loss   = loss_fn(y_pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f'Epoch {epoch+1} | avg loss: {avg_loss:.4f}')

## Train/Val split with DataLoader

In [ ]:
from torch.utils.data import random_split

dataset = RegressionDataset(1000)

# Split 80/20
train_set, val_set = random_split(dataset, [800, 200])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False)  # no shuffle for val

print('Train batches:', len(train_loader))
print('Val batches:  ', len(val_loader))